# 🎥 Feature Extraction with YOLOv8 Pose

This notebook implements the **video feature extraction pipeline** used in our study on hyperkinetic movement disorders.

* **Goal**: Convert raw clinical videos into structured **pose-based time series** and extract interpretable features (statistical, temporal, spectral, non-linear) for downstream machine learning.
* **Core method**: [YOLOv8 Pose](https://docs.ultralytics.com) (`yolov8x-pose-p6.pt`) is applied to detect 17 anatomical keypoints frame by frame.
* **Inputs**:

  * `video_dir/` → directory containing raw `.mp4` videos.
  * `yolov8x-pose-p6.pt` → pretrained YOLOv8 Pose model.
* **Outputs**:

  * `outputs/` → structured datasets (`.csv`/`.xlsx`) with extracted pose coordinates, distances, and derived features.

⚡ **Recommended environment**: Run on **Google Colab** with an Nvidia GPU (CUDA).


In [17]:
# Import Google Colab's drive module and Mount Google Drive at the given path (/content/drive).
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
# Install the Ultralytics package directly from PyPI.
# Ultralytics provides the YOLOv8 framework, including YOLOv8-Pose,
# which is used here for human pose estimation (17 anatomical keypoints).
# This command will download and install the latest stable release.

!pip install ultralytics

In [19]:
# OpenCV library for video and image processing
import cv2

# Standard Python library for measuring execution time (optional, useful for profiling)
import time

# Standard Python math library (imported here with alias 'm' for convenience)
import math as m

# (Optional) Google's Mediapipe library for alternative pose/landmark detection
# Not used in this notebook, but left here as a reference
# import mediapipe as mp

# Matplotlib for plotting and visualizations
import matplotlib.pyplot as plt

# NumPy for numerical operations (arrays, linear algebra, etc.)
import numpy as np

# Standard library for operating system interfaces (directory, file management)
import os

# Ultralytics library, provides YOLOv8 models (we use YOLOv8-Pose)
from ultralytics import YOLO

# Pandas for structured data manipulation (DataFrames, CSV/Excel exports)
import pandas as pd

# SciPy spatial module, here used for distance calculations between keypoints
from scipy.spatial import distance

# Pydantic for defining structured data models (validation and type safety)
from pydantic import BaseModel

# NumPy imported again (redundant, already imported above)
import numpy as np


In [20]:
# Path to the folder containing the input videos.
# In this case, the videos are stored on Google Drive under: "MyDrive/Colab Notebooks/videos_dir"
# ⚠️ Make sure this folder exists in your Drive and contains .mp4 files to process.
video_dir = "/content/drive/MyDrive/Colab Notebooks/videos_dir"


In [21]:
# List all files inside the input video directory
# This will return a list of filenames (e.g., ["video1.mp4", "video2.mp4", ...])
video_list = os.listdir(video_dir)

# Display the list of video files to verify that the directory was read correctly
video_list

['20240703_105018.mp4']

## Member Detection Function

In [25]:
# Define a structured model for YOLOv8-Pose keypoints.
# Each attribute corresponds to one anatomical landmark, with its index
# in the 17-keypoint COCO format used by YOLOv8-Pose.
# Using Pydantic's BaseModel makes the structure explicit and type-safe.

class GetKeypoint(BaseModel):
    NOSE:           int = 0
    LEFT_EYE:       int = 1
    RIGHT_EYE:      int = 2
    LEFT_EAR:       int = 3
    RIGHT_EAR:      int = 4
    LEFT_SHOULDER:  int = 5
    RIGHT_SHOULDER: int = 6
    LEFT_ELBOW:     int = 7
    RIGHT_ELBOW:    int = 8
    LEFT_WRIST:     int = 9
    RIGHT_WRIST:    int = 10
    LEFT_HIP:       int = 11
    RIGHT_HIP:      int = 12
    LEFT_KNEE:      int = 13
    RIGHT_KNEE:     int = 14
    LEFT_ANKLE:     int = 15
    RIGHT_ANKLE:    int = 16

In [26]:
# Define the path to the YOLOv8-Pose model weights.
# The file 'yolov8x-pose-p6.pt' is a pretrained YOLOv8 model specialized for pose estimation
# (17 anatomical keypoints). It must be located in your Google Drive at the given path.
# ⚠️ Make sure the file exists before running the model.
model_path = '/content/drive/MyDrive/Colab Notebooks/yolov8x-pose-p6.pt'


In [32]:
def member_tremor(file_name, model_path):
    """
    Process a video with YOLOv8-Pose to extract keypoints and save:
    - An annotated video with landmarks
    - A time series dataset of keypoint coordinates and distances
    """

    ## ---------------------------
    ## Load video
    ## ---------------------------
    cap = cv2.VideoCapture(file_name)
    print("read video")

    # Extract metadata (FPS, width, height, frame size)
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_size = (width, height)

    # Define video codec
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    # Extract video ID from filename
    base = os.path.basename(file_name)
    video_id = os.path.splitext(base)[0]

    # Define video output path
    file_name_output = "/content/drive/MyDrive/Colab Notebooks/outputs/" + video_id + '_yolo.mp4'

    # Initialize video writer
    video_output = cv2.VideoWriter(file_name_output, fourcc, fps, frame_size)

    import torch

    ## ---------------------------
    ## Device configuration
    ## ---------------------------
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Running on device:", device)

    # Load YOLOv8-Pose model
    model = YOLO(model_path)
    model = model.to(device)

    # Display model device
    print("model is running on :" + str(model.device))

    ## ---------------------------
    ## Prepare dataframe for keypoints
    ## ---------------------------
    get_keypoint = GetKeypoint()  # function assumed to return list of keypoints
    colonne = ['time']

    # Create column names: one pair (x,y) per keypoint
    for point in get_keypoint:
        colonne.append(point[0].lower() + '_x')
        colonne.append(point[0].lower() + '_y')

    df = pd.DataFrame(columns=colonne)

    ## ---------------------------
    ## Process video frame by frame
    ## ---------------------------
    print('compute video')
    while cap.isOpened():
        # Capture frame
        success, image = cap.read()
        if not success:
            print("Null.Frames")
            break

        # Run YOLO model inference on frame
        results_img = model(image, verbose=False)

        # Extract keypoints (normalized x,y in range [0,1])
        #result_keypoint = results_img[0].keypoints.xyn.cpu().numpy()[0]
        # Secure keypoint extraction
        result_keypoint = None
        if results_img[0].keypoints is not None and results_img[0].keypoints.xyn is not None:
          keypoints_array = results_img[0].keypoints.xyn.cpu().numpy()
          if keypoints_array.shape[0] > 0:  # At least one person detected
            result_keypoint = keypoints_array[0]



        # Calculate timestamp for current frame
        timestamps = round(cap.get(cv2.CAP_PROP_POS_MSEC) / 1000, 2)

        # Get list of keypoints again
        get_keypoint = GetKeypoint()

        # Build one row of results
        #row = {}
        #row['time'] = timestamps
        #for point in get_keypoint:
        #    if len(point) > 0:
        #        try:
        #            row[point[0].lower() + '_x'] = int(result_keypoint[point[1]][0] * width)
        #            row[point[0].lower() + '_y'] = int(result_keypoint[point[1]][1] * height)
        #        except:
        #            row[point[0].lower() + '_x'] = np.nan
        #            row[point[0].lower() + '_y'] = np.nan
        #    else:
        #        row[point[0].lower() + '_x'] = np.nan
        #        row[point[0].lower() + '_y'] = np.nan

        row = {}
        row['time'] = timestamps
        for point in get_keypoint:
          if result_keypoint is not None:  # only if detection exists
            try:
              row[point[0].lower() + '_x'] = int(result_keypoint[point[1]][0] * width)
              row[point[0].lower() + '_y'] = int(result_keypoint[point[1]][1] * height)
            except:
              row[point[0].lower() + '_x'] = np.nan
              row[point[0].lower() + '_y'] = np.nan
          else:
            row[point[0].lower() + '_x'] = np.nan
            row[point[0].lower() + '_y'] = np.nan




        # Append row to dataframe
        df.loc[len(df)] = row

        # Draw landmarks on the frame
        results_img = results_img[0].plot()

        # Write frame with landmarks to output video
        video_output.write(results_img)

    # Release resources
    print("video " + video_id + " treated successfully and stored in " + file_name_output)
    cap.release()
    video_output.release()

    ## ---------------------------
    ## Calculate distances from (0,0)
    ## ---------------------------
    for point in get_keypoint:
        try:
            df[point[0].lower() + '_distance'] = df[
                [point[0].lower() + '_x', point[0].lower() + '_y']
            ].apply(lambda x: distance.euclidean([0, 0, 0], [x[0], x[1], 0]), axis=1)
        except:
            df[point[0].lower() + '_distance'] = np.nan
            print(point[0] + '_error')

    ## ---------------------------
    ## Save results to Excel
    ## ---------------------------
    df.to_excel(
        '/content/drive/MyDrive/Colab Notebooks/outputs/' + video_id + '_timeseries.xlsx',
        index=False
    )

    print("tremor dataset for the video " + video_id + " treated successfully and stored in " + video_id + '_timeseries.xlsx')
    print('end')


## Execute Prediction For All Videos

In [33]:
# Measure the execution time of the whole loop
%%time

# Iterate over all videos in the input directory
for video in video_list:
    # Build full path to the video file
    file_name = video_dir + '/' + video

    # Start processing message
    print("start : " + file_name)

    # Run the feature extraction function on this video
    member_tremor(file_name, model_path)

    # End processing message
    print("end : " + file_name)


start : /content/drive/MyDrive/Colab Notebooks/videos_dir/20240703_105018.mp4
read video
Running on device: cuda
model is running on :cuda:0
compute video
Null.Frames
video 20240703_105018 treated successfully and stored in /content/drive/MyDrive/Colab Notebooks/outputs/20240703_105018_yolo.mp4


/tmp/ipython-input-2901186336.py:145: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ].apply(lambda x: distance.euclidean([0, 0, 0], [x[0], x[1], 0]), axis=1)


NOSE_error


/tmp/ipython-input-2901186336.py:145: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ].apply(lambda x: distance.euclidean([0, 0, 0], [x[0], x[1], 0]), axis=1)


LEFT_EYE_error


...

RIGHT_KNEE_error


/tmp/ipython-input-2901186336.py:145: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ].apply(lambda x: distance.euclidean([0, 0, 0], [x[0], x[1], 0]), axis=1)


LEFT_ANKLE_error


/tmp/ipython-input-2901186336.py:145: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  ].apply(lambda x: distance.euclidean([0, 0, 0], [x[0], x[1], 0]), axis=1)


RIGHT_ANKLE_error
tremor dataset for the video 20240703_105018 treated successfully and stored in 20240703_105018_timeseries.xlsx
end
end : /content/drive/MyDrive/Colab Notebooks/videos_dir/20240703_105018.mp4
CPU times: user 49min 11s, sys: 5.71 s, total: 49min 17s
Wall time: 41min 39s
